# DINOv3 ViT-B + VPT MLP — Contrail Segmentation

Run this notebook on **Kaggle** (recommended — dataset already available, free T4 GPU)  
or on **Colab Pro+** (requires ~350 GB disk + Kaggle API download).

**Preprocessing notes:**
- `mask_only=True` → 3-channel false-color image at frame 4 (the labeled frame)
- DINOv3 normalization applied via `AutoImageProcessor` (exact stats the backbone was pretrained with)

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Clone repo & install dependencies

In [ ]:
!git clone https://github.com/aryangarg794/contrail-segmentation.git
%cd contrail-segmentation
!git checkout attention-unet-training

In [ ]:
!pip install -q torch torchvision lightning "transformers>=5.3.0" timm \
    albumentations segmentation-models-pytorch \
    hydra-core wandb dill rich tqdm
!pip install -q -e .

import sys, importlib
sys.path.insert(0, '/kaggle/working/contrail-segmentation/src')
importlib.invalidate_caches()

## 3. Dataset setup

### Option A — Kaggle (recommended)
Add the competition dataset via the Kaggle sidebar:  
`+ Add Data → google-research-identify-contrails-reduce-global-warming`  
Data will be at `/kaggle/input/google-research-identify-contrails-reduce-global-warming/`

### Option B — Google Colab Pro+
Uncomment the Colab block below and upload your `kaggle.json`.

In [ ]:
import os

# --- Option A: Kaggle ---
DATA_ROOT = '/kaggle/input/competitions/google-research-identify-contrails-reduce-global-warming'

# --- Option B: Colab Pro+ ---
# from google.colab import files
# files.upload()  # upload kaggle.json
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle competitions download -c google-research-identify-contrails-reduce-global-warming -p /content/data
# !unzip -q /content/data/google-research-identify-contrails-reduce-global-warming.zip -d /content/data
# DATA_ROOT = '/content/data'

TRAIN_DIR = os.path.join(DATA_ROOT, 'train')
META_PATH = os.path.join(DATA_ROOT, 'train_metadata.json')
print('Train dir exists:', os.path.exists(TRAIN_DIR))
print('Metadata exists: ', os.path.exists(META_PATH))

## 4. Patch data paths at runtime

In [ ]:
import pandas as pd
import contrail_segmentation.data.utils as data_utils

data_utils.DATA_DIR  = TRAIN_DIR
data_utils.META_PATH = META_PATH
data_utils.metadata  = pd.read_json(META_PATH)
print(f'Dataset size: {len(data_utils.metadata)} records')

## 5. W&B login

**Kaggle:** add your W&B API key under *Settings → Secrets* with the name `WANDB_API_KEY`, then run the Kaggle cell.  
**Colab:** run the Colab cell — it will prompt you to paste your key.

In [ ]:
import wandb

# --- Kaggle ---
from kaggle_secrets import UserSecretsClient
wandb.login(key=UserSecretsClient().get_secret('WANDB_API_KEY'))

# --- Colab ---
# wandb.login()

## 6. Build preprocessing transforms

We fetch the exact normalization stats DINOv3 was pretrained with via `AutoImageProcessor`,  
then pass them to albumentations `A.Normalize` so the frozen backbone sees in-distribution inputs.

In [ ]:
import albumentations as A
from transformers import AutoImageProcessor

MODEL_NAME = 'facebook/dinov3-vitb16-pretrain-lvd1689m'

processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
mean, std = processor.image_mean, processor.image_std
print(f'DINOv3 norm  mean={mean}  std={std}')

# Training: augmentation + normalization
train_transform = A.Compose([
    A.ShiftScaleRotate(scale_limit=0.2, rotate_limit=180, shift_limit=0.3,
                       border_mode=0, value=0, p=0.5),
    A.HorizontalFlip(),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.3, p=0.5),
    A.Normalize(mean=mean, std=std),
])

# Validation: normalization only
val_transform = A.Compose([
    A.Normalize(mean=mean, std=std),
])

## 7. Build dataloaders

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from contrail_segmentation.data.dataset import ContrailDataset

SEED        = 0
BATCH_SIZE  = 16
NUM_WORKERS = 2  # keep low on Kaggle/Colab

torch.manual_seed(SEED)
np.random.seed(SEED)
generator = torch.Generator().manual_seed(SEED)

# mask_only=True: 3-channel false-color at frame 4 (the labeled frame) — required for DINOv3
full_dataset = ContrailDataset(mask_only=True)
indices      = np.arange(len(full_dataset))
np.random.shuffle(indices)
train_size   = int(0.8 * len(indices))
train_idx, val_idx = indices[:train_size], indices[train_size:]

train_set = ContrailDataset(mask_only=True, transform=train_transform)
val_set   = ContrailDataset(mask_only=True, transform=val_transform)

train_loader = DataLoader(Subset(train_set, train_idx), batch_size=BATCH_SIZE,
                          shuffle=True, generator=generator,
                          pin_memory=True, num_workers=NUM_WORKERS)
val_loader   = DataLoader(Subset(val_set, val_idx), batch_size=BATCH_SIZE,
                          pin_memory=True, num_workers=NUM_WORKERS)

print(f'Train: {len(train_idx)} | Val: {len(val_idx)}')

## 8. Build model

In [ ]:
from contrail_segmentation.models.dino_mlp import DINOv3MLP

model = DINOv3MLP(
    model_name=MODEL_NAME,
    num_vpt=50,
    lr=1e-4,
    wd=1e-3,
    threshold=0.5,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} params ({100*trainable/total:.1f}%)')

## 9. Train

In [ ]:
from datetime import datetime
from lightning.pytorch import Trainer
from lightning.pytorch.loggers import WandbLogger
from contrail_segmentation.train.utils import find_best_threshold

MAX_EPOCHS = 100
timestamp  = datetime.now().strftime('%d_%b_%Y__%Hh%Mm')
run_name   = f'dinov3_vitb_vpt_mlp_seed{SEED}_{timestamp}'

logger = WandbLogger(project='contrail-segmentation', name=run_name, save_dir='wandb_logs')

trainer = Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator='gpu',
    devices=1,
    precision='16-mixed',
    log_every_n_steps=1,
    logger=logger,
)

trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

## 10. Test

In [ ]:
best_thresh = find_best_threshold(model, val_loader)
model.threshold = best_thresh
model.mask_only = True

test_metrics = trainer.test(model, dataloaders=val_loader)
print(test_metrics)

## 11. Save checkpoint

In [ ]:
torch.save(model.state_dict(), f'dinov3_vitb_vpt_mlp_{timestamp}.pt')
print('Saved.')